In [22]:
"""Environment configuration loader with interactive setup and dependency management."""
import os
import sys
import subprocess
from pathlib import Path
from getpass import getpass
from typing import List, Optional, Dict
from importlib import import_module

# Add project paths to Python search path
sys.path.append(os.path.abspath("src"))
sys.path.append(os.path.abspath("agents"))


class DependencyManager:
    """Manages package dependencies with auto-installation."""
    
    REQUIRED_PACKAGES = {
        "dotenv": "python-dotenv",
        "litellm": "litellm",
    }
    
    @classmethod
    def check_and_install(cls) -> None:
        """Check for missing packages and install them."""
        missing = cls._get_missing_packages()
        
        if missing:
            cls._install_packages(missing)
    
    @classmethod
    def _get_missing_packages(cls) -> List[str]:
        """Identify missing packages.
        
        Returns:
            List of pip package names that need installation
        """
        missing = []
        
        for import_name, pip_name in cls.REQUIRED_PACKAGES.items():
            try:
                import_module(import_name)
            except ImportError:
                missing.append(pip_name)
        
        return missing
    
    @classmethod
    def _install_packages(cls, packages: List[str]) -> None:
        """Install missing packages via pip.
        
        Args:
            packages: List of pip package names to install
        """
        print(f"Installing dependencies: {', '.join(packages)}")
        
        for package in packages:
            try:
                subprocess.check_call(
                    [sys.executable, "-m", "pip", "install", package],
                    stdout=subprocess.DEVNULL,
                    stderr=subprocess.PIPE
                )
            except subprocess.CalledProcessError as e:
                raise RuntimeError(f"Failed to install {package}: {e}")
        
        print()


class EnvironmentConfig:
    """Manages loading and validation of environment variables."""
    
    REQUIRED_VARS = [
        "OPENROUTER_API_KEY",
        "RUNPOD_API_KEY",
        "RUNPOD_ENDPOINT_ID"
    ]
    
    def __init__(self, env_path: Optional[Path] = None):
        """Initialize configuration loader.
        
        Args:
            env_path: Path to .env file. Defaults to ~/workspace/.env
        """
        self.env_path = env_path or Path.home() / "workspace" / ".env"
        self._load_environment()
    
    def _load_environment(self) -> None:
        """Load environment variables from .env file."""
        from dotenv import load_dotenv
        
        load_dotenv(self.env_path, override=True)
        
        # Display path context
        cwd = Path.cwd()
        path_parts = cwd.parts[-4:] if len(cwd.parts) >= 4 else cwd.parts
        print(f"Working directory: .../{'/'.join(path_parts)}")
        print(f"Environment source: {self.env_path}\n")
    
    def _prompt_for_missing_var(self, var_name: str) -> None:
        """Prompt user for missing environment variable.
        
        Args:
            var_name: Name of the environment variable
        """
        print(f"WARNING: {var_name} not found in .env file.")
        print("Please review the README or .env.example for setup instructions.")
        
        user_input = getpass(f"Please enter value for {var_name}: ")
        
        if user_input.strip():
            os.environ[var_name] = user_input.strip()
        else:
            print(f"Skipping {var_name} (no input provided)...")
    
    def _display_var_status(self, var_name: str) -> None:
        """Display the status of an environment variable.
        
        Args:
            var_name: Name of the environment variable
        """
        value = os.getenv(var_name)
        
        if value:
            # Show first 5 and last 4 characters
            masked = f"{value[:5]}...{value[-4:]}" if len(value) > 9 else f"{value[:5]}..."
            print(f"{var_name:<25}: {masked}")
        else:
            print(f"{var_name:<25}: MISSING")
    
    def validate_and_setup(self) -> None:
        """Validate required variables and prompt for missing ones."""
        for var in self.REQUIRED_VARS:
            if not os.getenv(var):
                self._prompt_for_missing_var(var)
            
            self._display_var_status(var)
        
        print("-" * 40)
        self._raise_if_missing()
    
    def _raise_if_missing(self) -> None:
        """Raise ValueError if any required variables are missing."""
        missing = [var for var in self.REQUIRED_VARS if not os.getenv(var)]
        
        if missing:
            raise ValueError(
                f"Critical configuration missing: {', '.join(missing)}"
            )
    
    def get_config_dict(self) -> Dict[str, Optional[str]]:
        """Get all required variables as a dictionary.
        
        Returns:
            Dictionary mapping variable names to their values
        """
        return {var: os.getenv(var) for var in self.REQUIRED_VARS}
    
    @classmethod
    def setup(cls, env_path: Optional[Path] = None, check_deps: bool = True) -> "EnvironmentConfig":
        """Convenience method to load and validate configuration.
        
        Args:
            env_path: Path to .env file. Defaults to ~/workspace/.env
            check_deps: Whether to check and install dependencies
            
        Returns:
            Configured EnvironmentConfig instance
            
        Raises:
            ValueError: If required variables are missing after prompting
        """
        if check_deps:
            DependencyManager.check_and_install()
        
        config = cls(env_path)
        config.validate_and_setup()
        return config


def verify_litellm_setup() -> None:
    """Verify LiteLLM can connect with current environment."""
    import litellm
    from litellm import completion
    
    response = completion(
        model="openrouter/z-ai/glm-4.6",
        messages=[{"role": "user", "content": "Reply with: Connection successful"}],
        max_tokens=20
    )
    
    print(f"LiteLLM connection verified: {response.choices[0].message.content}")


# Usage
if __name__ == "__main__":
    config = EnvironmentConfig.setup()
    
    try:
        verify_litellm_setup()
    except Exception as e:
        print(f"LiteLLM verification failed: {e}")

23:16:50 - LiteLLM:DEBUG: utils.py:381 - 

23:16:50 - LiteLLM:DEBUG: utils.py:381 - Request to litellm:
23:16:50 - LiteLLM:DEBUG: utils.py:381 - litellm.completion(model='openrouter/z-ai/glm-4.6', messages=[{'role': 'user', 'content': 'Reply with: Connection successful'}], max_tokens=20)
23:16:50 - LiteLLM:DEBUG: utils.py:381 - 

23:16:50 - LiteLLM:DEBUG: litellm_logging.py:509 - self.optional_params: {}
23:16:50 - LiteLLM:DEBUG: utils.py:381 - SYNC kwargs[caching]: False; litellm.cache: None; kwargs.get('cache')['no-cache']: False
23:16:50 - LiteLLM:INFO: utils.py:3419 - 
LiteLLM completion() model= z-ai/glm-4.6; provider = openrouter
23:16:50 - LiteLLM:DEBUG: utils.py:3422 - 
LiteLLM: Params passed to completion() {'model': 'z-ai/glm-4.6', 'functions': None, 'function_call': None, 'temperature': None, 'top_p': None, 'n': None, 'stream': None, 'stream_options': None, 'stop': None, 'max_tokens': 20, 'max_completion_tokens': None, 'modalities': None, 'prediction': None, 'audio': None, '

Working directory: .../home/jovyan/workspace/notebooks
Environment source: /home/jovyan/workspace/.env

OPENROUTER_API_KEY       : sk-or...4d01
RUNPOD_API_KEY           : rpa_6...o1s7
RUNPOD_ENDPOINT_ID       : a48mr...g35n
----------------------------------------


23:16:53 - LiteLLM:DEBUG: litellm_logging.py:1121 - RAW RESPONSE:

         

         

         

         

         

         
{"id":"gen-1764908210-KbWHSFkXnP1FdFDNQwbY","provider":"Chutes","model":"z-ai/glm-4.6","object":"chat.completion","created":1764908210,"choices":[{"logprobs":null,"finish_reason":"length","native_finish_reason":"length","index":0,"message":{"role":"assistant","content":"","refusal":null,"reasoning":"\n1.  **Analyze the user's request:** The user has provided a very specific","reasoning_details":[{"format":"unknown","index":0,"type":"reasoning.text","text":"\n1.  **Analyze the user's request:** The user has provided a very specific"}]}}],"usage":{"prompt_tokens":10,"completion_tokens":20,"total_tokens":30,"cost":0,"is_byok":true,"prompt_tokens_details":{"cached_tokens":4,"audio_tokens":0,"video_tokens":0},"cost_details":{"upstream_inference_cost":0.000039,"upstream_inference_prompt_cost":0.000004,"upstream_inference_completions_cost":0.000035},"completion_t

LiteLLM connection verified: 


23:16:53 - LiteLLM:DEBUG: litellm_logging.py:1801 - Logging Details LiteLLM-Success Call: Cache_hit=None
23:16:53 - LiteLLM:DEBUG: utils.py:4830 - checking potential_model_names in litellm.model_cost: {'split_model': 'z-ai/glm-4.6', 'combined_model_name': 'openrouter/z-ai/glm-4.6', 'stripped_model_name': 'z-ai/glm-4.6', 'combined_stripped_model_name': 'openrouter/z-ai/glm-4.6', 'custom_llm_provider': 'openrouter'}
23:16:53 - LiteLLM:DEBUG: utils.py:5171 - model_info: {'key': 'openrouter/z-ai/glm-4.6', 'max_tokens': 202800, 'max_input_tokens': 202800, 'max_output_tokens': 131000, 'input_cost_per_token': 4e-07, 'input_cost_per_token_flex': None, 'input_cost_per_token_priority': None, 'cache_creation_input_token_cost': None, 'cache_creation_input_token_cost_above_200k_tokens': None, 'cache_read_input_token_cost': None, 'cache_read_input_token_cost_above_200k_tokens': None, 'cache_read_input_token_cost_flex': None, 'cache_read_input_token_cost_priority': None, 'cache_creation_input_token_c

In [23]:
# %% [markdown]
# # RunPod ComfyUI Image Generation with Google ADK
# 
# This notebook demonstrates how to use a RunPod-hosted ComfyUI workflow
# as a tool for Google ADK agents to generate images from text prompts.

# %% [markdown]
# ## Setup and Imports

# %%
import os
import json
import time
import random
import asyncio
from typing import Dict, Optional

import requests
from dotenv import load_dotenv

from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# Load environment variables
load_dotenv()

print("Imports successful")

# %% [markdown]
# ## Configure API Keys
# 
# Required environment variables:
# - RUNPOD_API_KEY
# - RUNPOD_ENDPOINT_ID (optional, defaults to a48mrbdsbzg35n)
# - OPENROUTER_API_KEY

# %%
# Verify API keys are configured
RUNPOD_API_KEY = os.getenv("RUNPOD_API_KEY")
RUNPOD_ENDPOINT_ID = os.getenv("RUNPOD_ENDPOINT_ID", "a48mrbdsbzg35n")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not RUNPOD_API_KEY:
    print("ERROR: RUNPOD_API_KEY not set")
else:
    print("RUNPOD_API_KEY configured")

if not OPENROUTER_API_KEY:
    print("ERROR: OPENROUTER_API_KEY not set")
else:
    print("OPENROUTER_API_KEY configured")

print(f"Using RunPod endpoint: {RUNPOD_ENDPOINT_ID}")

# %% [markdown]
# ## Define the RunPod Image Generation Tool

# %%
def generate_image_with_runpod(
    prompt: str,
    width: int = 1152,
    height: int = 2048,
    steps: int = 4,
    cfg: float = 1.0,
    seed: Optional[int] = None,
) -> Dict[str, str]:
    """
    Generates an image using the RunPod Z-Image Turbo workflow.

    Args:
        prompt: The text description of the image to generate
        width: Image width in pixels (default: 1152)
        height: Image height in pixels (default: 2048)
        steps: Number of generation steps 1-20 (default: 4)
        cfg: Guidance scale 0.5-2.0 (default: 1.0)
        seed: Random seed for reproducibility (optional)

    Returns:
        Dictionary with status, message, and image data or error details
    """
    # Get API credentials
    api_key = os.getenv("RUNPOD_API_KEY")
    if not api_key:
        return {
            "status": "error",
            "message": "RunPod API key not configured",
            "error": "RUNPOD_API_KEY environment variable is not set",
        }

    endpoint_id = os.getenv("RUNPOD_ENDPOINT_ID", "a48mrbdsbzg35n")

    # Generate random seed if not provided
    if seed is None:
        seed = random.randint(0, 2**32 - 1)

    # Construct the ComfyUI workflow payload
    workflow_payload = {
        "input": {
            "workflow": {
                "9": {
                    "inputs": {
                        "filename_prefix": "Z-Image\\ComfyUI",
                        "images": ["43", 0],
                    },
                    "class_type": "SaveImage",
                    "_meta": {"title": "Save Image"},
                },
                "39": {
                    "inputs": {
                        "clip_name": "qwen_3_4b.safetensors",
                        "type": "lumina2",
                        "device": "default",
                    },
                    "class_type": "CLIPLoader",
                    "_meta": {"title": "Load CLIP"},
                },
                "40": {
                    "inputs": {"vae_name": "ae.safetensors"},
                    "class_type": "VAELoader",
                    "_meta": {"title": "Load VAE"},
                },
                "41": {
                    "inputs": {
                        "width": width,
                        "height": height,
                        "batch_size": 1,
                    },
                    "class_type": "EmptySD3LatentImage",
                    "_meta": {"title": "EmptySD3LatentImage"},
                },
                "42": {
                    "inputs": {"conditioning": ["45", 0]},
                    "class_type": "ConditioningZeroOut",
                    "_meta": {"title": "ConditioningZeroOut"},
                },
                "43": {
                    "inputs": {
                        "samples": ["44", 0],
                        "vae": ["40", 0],
                    },
                    "class_type": "VAEDecode",
                    "_meta": {"title": "VAE Decode"},
                },
                "44": {
                    "inputs": {
                        "seed": seed,
                        "steps": steps,
                        "cfg": cfg,
                        "sampler_name": "res_multistep",
                        "scheduler": "simple",
                        "denoise": 1,
                        "model": ["48", 0],
                        "positive": ["45", 0],
                        "negative": ["42", 0],
                        "latent_image": ["41", 0],
                    },
                    "class_type": "KSampler",
                    "_meta": {"title": "KSampler"},
                },
                "45": {
                    "inputs": {
                        "text": prompt,
                        "clip": ["39", 0],
                    },
                    "class_type": "CLIPTextEncode",
                    "_meta": {"title": "CLIP Text Encode (Prompt)"},
                },
                "48": {
                    "inputs": {
                        "unet_name": "z_image_turbo_bf16.safetensors",
                        "weight_dtype": "fp8_e4m3fn",
                    },
                    "class_type": "UNETLoader",
                    "_meta": {"title": "Unet Loader"},
                },
            }
        }
    }

    # Configure API endpoints
    run_url = f"https://api.runpod.ai/v2/{endpoint_id}/run"
    status_url_template = f"https://api.runpod.ai/v2/{endpoint_id}/status/"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

    try:
        # Submit the job to RunPod
        print("Submitting job to RunPod...")
        response = requests.post(
            run_url, headers=headers, json=workflow_payload, timeout=30
        )

        if response.status_code != 200:
            return {
                "status": "error",
                "message": f"Failed to submit job to RunPod (HTTP {response.status_code})",
                "error": response.text,
            }

        response_data = response.json()
        job_id = response_data.get("id")

        if not job_id:
            return {
                "status": "error",
                "message": "RunPod did not return a job ID",
                "error": json.dumps(response_data),
            }

        print(f"Job submitted with ID: {job_id}")
        print("Polling for completion...")

        # Poll for job completion
        status_url = status_url_template + job_id
        max_polls = 60  # 5 minutes max
        poll_count = 0

        while poll_count < max_polls:
            poll_count += 1
            time.sleep(5)

            status_response = requests.get(status_url, headers=headers, timeout=30)
            status_data = status_response.json()
            job_status = status_data.get("status")

            if poll_count % 6 == 0:  # Print every 30 seconds
                print(f"Still waiting... ({poll_count * 5}s elapsed)")

            if job_status == "COMPLETED":
                print("Job completed successfully")
                # Extract and return image data
                try:
                    output_image_data = status_data["output"]["images"][0]
                    image_base64 = output_image_data["data"]
                    filename = output_image_data["filename"]

                    return {
                        "status": "success",
                        "message": f"Image generated successfully: {filename}",
                        "filename": filename,
                        "image_data": image_base64,
                        "seed": seed,
                        "prompt": prompt,
                    }
                except (KeyError, IndexError, TypeError) as e:
                    return {
                        "status": "error",
                        "message": "Job completed but could not parse image data",
                        "error": str(e),
                    }

            elif job_status == "FAILED":
                return {
                    "status": "error",
                    "message": "RunPod job failed",
                    "error": status_data.get("error", "No error details provided"),
                }

            elif job_status not in ["IN_QUEUE", "IN_PROGRESS"]:
                return {
                    "status": "error",
                    "message": f"Unexpected job status: {job_status}",
                    "error": json.dumps(status_data),
                }

        # Job timed out
        return {
            "status": "error",
            "message": "Job timed out after 5 minutes",
            "error": "Maximum polling attempts reached",
        }

    except requests.exceptions.RequestException as e:
        return {
            "status": "error",
            "message": "Network error communicating with RunPod",
            "error": str(e),
        }
    except Exception as e:
        return {
            "status": "error",
            "message": "Unexpected error during image generation",
            "error": str(e),
        }


print("Image generation tool defined")

# %% [markdown]
# ## Create the ADK Agent
# 
# Model Options (choose one):
# - openrouter/anthropic/claude-3.5-sonnet (recommended, most reliable)
# - openrouter/google/gemini-2.0-flash-exp:free (free tier, may have rate limits)
# - openrouter/openai/gpt-4o-mini (cost-effective)
# - openrouter/meta-llama/llama-3.1-8b-instruct:free (free tier alternative)

# %%
# Create the LiteLLM model with OpenRouter
# Using Claude Sonnet for reliability, change if needed
MODEL_NAME = "z-ai/glm-4.6"

print(f"Initializing model: {MODEL_NAME}")

try:
    model = LiteLlm(
        model=MODEL_NAME,
        api_key=OPENROUTER_API_KEY,
    )
    
    # Create the agent with the RunPod tool
    agent = LlmAgent(
        name="image_generator",
        model=model,
        instruction="""
        You are an AI assistant that can generate images using the RunPod service.
        When users ask you to create, generate, or make an image, use the 
        generate_image_with_runpod tool.
        
        After generating an image, inform the user about:
        - The filename where the image was saved
        - The seed used (so they can reproduce it)
        - Any relevant details about the generation
        
        If there's an error, explain it clearly and suggest what the user might do.
        Be enthusiastic and helpful about image generation.
        """,
        tools=[generate_image_with_runpod],
    )
    
    print("Agent created successfully")
    
except Exception as e:
    print(f"ERROR: Failed to create agent - {e}")
    raise

# %% [markdown]
# ## Run the Demo

# %%
async def run_demo():
    """
    Runs a demo of the image generation agent.
    """
    print("=" * 70)
    print("RunPod Image Generation Demo")
    print("=" * 70)

    # Initialize session service
    print("\n1. Setting up session...")
    session_service = InMemorySessionService()

    # Create session
    await session_service.create_session(
        user_id="demo_user",
        session_id="demo_session",
        app_name="runpod_image_gen",
    )
    print("Session created")

    # Initialize runner
    print("\n2. Initializing runner...")
    runner = Runner(
        agent=agent,
        app_name="runpod_image_gen",
        session_service=session_service,
    )
    print("Runner ready")

    # Prepare the prompt
    prompt_text = "Generate a cyberpunk gopher astronaut floating in space, digital art"
    print(f"\n3. Sending prompt: '{prompt_text}'")
    print("Note: This may take 30-60 seconds\n")

    content = types.Content(
        role="user",
        parts=[types.Part(text=prompt_text)],
    )

    # Run the agent
    print("4. Agent processing...\n")
    final_response = None

    try:
        async for event in runner.run_async(
            user_id="demo_user",
            session_id="demo_session",
            new_message=content,
        ):
            if event.is_final_response():
                final_response = event.content.parts[0].text
                break

        if final_response:
            print("\n" + "=" * 70)
            print("AGENT RESPONSE:")
            print("=" * 70)
            print(final_response)
            print("=" * 70)
        else:
            print("\nNo final response received")

    except Exception as e:
        error_message = str(e)
        print(f"\nError during execution: {error_message}")
        
        # Provide helpful guidance based on error type
        if "RateLimitError" in error_message or "429" in error_message:
            print("\nRate limit hit. Suggestions:")
            print("1. Wait a few minutes and try again")
            print("2. Change MODEL_NAME to: openrouter/anthropic/claude-3.5-sonnet")
            print("3. Add your own API key to OpenRouter for higher limits")
            print("   Visit: https://openrouter.ai/settings/integrations")
        elif "BadRequestError" in error_message:
            print("\nModel configuration error. Check that MODEL_NAME includes 'openrouter/' prefix")
        else:
            import traceback
            traceback.print_exc()

    print("\nDemo complete")


# Run the demo
await run_demo()

# %% [markdown]
# ## Alternative: Test Direct Function Call (No LLM)
# 
# If you want to test the RunPod integration without the LLM agent,
# you can call the function directly:

# %%
# Uncomment to test direct function call
# print("Testing direct RunPod API call...")
# result = generate_image_with_runpod(
#     prompt="a cute robot cat, digital art",
#     width=1024,
#     height=1024,
#     steps=4,
#     seed=12345
# )
# 
# print(f"\nStatus: {result['status']}")
# print(f"Message: {result['message']}")
# if result['status'] == 'success':
#     print(f"Seed: {result['seed']}")
#     print(f"Filename: {result['filename']}")
#     # To display the image in the notebook:
#     # import base64
#     # from IPython.display import Image, display
#     # image_bytes = base64.b64decode(result['image_data'])
#     # display(Image(image_bytes))

# %% [markdown]
# ## Troubleshooting
# 
# ### Rate Limits
# If you hit rate limits with free models, try:
# - openrouter/anthropic/claude-3.5-sonnet (more reliable)
# - openrouter/openai/gpt-4o-mini (cost-effective)
# - Add your own API keys to OpenRouter for higher limits
# 
# ### Model Configuration
# Model string must be: openrouter/<provider>/<model>
# 
# ### RunPod Issues
# - Check endpoint status at RunPod dashboard
# - Verify API key is valid
# - Ensure endpoint ID is correct
# 
# ### Display Generated Images
# Uncomment the image display code in the "Test Direct Function Call" section
# to view generated images directly in the notebook.

23:21:33 - LiteLLM:DEBUG: utils.py:381 - 

23:21:33 - LiteLLM:DEBUG: utils.py:381 - Request to litellm:
23:21:33 - LiteLLM:DEBUG: utils.py:381 - litellm.acompletion(model='z-ai/glm-4.6', messages=[{'role': 'system', 'content': '\n        You are an AI assistant that can generate images using the RunPod service.\n        When users ask you to create, generate, or make an image, use the \n        generate_image_with_runpod tool.\n        \n        After generating an image, inform the user about:\n        - The filename where the image was saved\n        - The seed used (so they can reproduce it)\n        - Any relevant details about the generation\n        \n        If there\'s an error, explain it clearly and suggest what the user might do.\n        Be enthusiastic and helpful about image generation.\n        \n\nYou are an agent. Your internal name is "image_generator".'}, {'role': 'user', 'content': 'Generate a cyberpunk gopher astronaut floating in space, digital art'}], tools=[{'ty

Imports successful
RUNPOD_API_KEY configured
OPENROUTER_API_KEY configured
Using RunPod endpoint: a48mrbdsbzg35n
Image generation tool defined
Initializing model: z-ai/glm-4.6
Agent created successfully
RunPod Image Generation Demo

1. Setting up session...
Session created

2. Initializing runner...
Runner ready

3. Sending prompt: 'Generate a cyberpunk gopher astronaut floating in space, digital art'
Note: This may take 30-60 seconds

4. Agent processing...


Provider List: https://docs.litellm.ai/docs/providers


Error during execution: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=z-ai/glm-4.6
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers

Model configuration error. Check that MODEL_NAME includes 'openrouter/' prefix

Demo complete


In [12]:
import litellm
litellm._turn_on_debug()  # Prints full request/response for troubleshooting

from litellm import completion
import os
os.environ["OPENROUTER_API_KEY"] = "your_key_here"  # Or just rely on .env
response = completion(
    model="z-ai/glm-4.6",
    messages=[{"role": "user", "content": "Say hi!"}]
)
print(response.choices[0].message.content)

22:57:31 - LiteLLM:DEBUG: utils.py:381 - 

22:57:31 - LiteLLM:DEBUG: utils.py:381 - Request to litellm:
22:57:31 - LiteLLM:DEBUG: utils.py:381 - litellm.completion(model='openrouter/z-ai/glm-4.6', messages=[{'role': 'user', 'content': 'Say hi!'}])
22:57:31 - LiteLLM:DEBUG: utils.py:381 - 

22:57:31 - LiteLLM:DEBUG: litellm_logging.py:509 - self.optional_params: {}
22:57:31 - LiteLLM:DEBUG: utils.py:381 - SYNC kwargs[caching]: False; litellm.cache: None; kwargs.get('cache')['no-cache']: False
22:57:31 - LiteLLM:INFO: utils.py:3419 - 
LiteLLM completion() model= z-ai/glm-4.6; provider = openrouter
22:57:31 - LiteLLM:DEBUG: utils.py:3422 - 
LiteLLM: Params passed to completion() {'model': 'z-ai/glm-4.6', 'functions': None, 'function_call': None, 'temperature': None, 'top_p': None, 'n': None, 'stream': None, 'stream_options': None, 'stop': None, 'max_tokens': None, 'max_completion_tokens': None, 'modalities': None, 'prediction': None, 'audio': None, 'presence_penalty': None, 'frequency_pen


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers



AuthenticationError: litellm.AuthenticationError: AuthenticationError: OpenrouterException - {"error":{"message":"No cookie auth credentials found","code":401}}